In [ ]:
!pip install datasets pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [ ]:
# PrimeKG 有disease_phenotype_positiv edge，不需要额外的HPO input
edges = pd.read_csv('edges.csv')
print("EDGES shape:", edges.shape)
print("EDGES columns:", edges.columns.tolist())
print("\nRelation types:")
print(edges['relation'].value_counts())

EDGES shape: (8100498, 4)
EDGES columns: ['relation', 'display_relation', 'x_index', 'y_index']

Relation types:
relation
anatomy_protein_present       3036406
drug_drug                     2672628
protein_protein                642150
disease_phenotype_positive     300634
bioprocess_protein             289610
cellcomp_protein               166804
disease_protein                160822
molfunc_protein                139060
drug_effect                    129568
bioprocess_bioprocess          105772
pathway_protein                 85292
disease_disease                 64388
contraindication                61350
drug_protein                    51306
anatomy_protein_absent          39774
phenotype_phenotype             37472
anatomy_anatomy                 28064
molfunc_molfunc                 27148
indication                      18776
cellcomp_cellcomp                9690
phenotype_protein                6660
off-label use                    5136
pathway_pathway                  5070
expo

In [ ]:
nodes = pd.read_csv('nodes.csv')
print(nodes.shape)
print(nodes.columns.tolist())
print(nodes['node_type'].value_counts())

(129375, 5)
['node_index', 'node_id', 'node_type', 'node_name', 'node_source']
node_type
biological_process    28642
gene/protein          27671
disease               17080
effect/phenotype      15311
anatomy               14035
molecular_function    11169
drug                   7957
cellular_component     4176
pathway                2516
exposure                818
Name: count, dtype: int64


In [ ]:
kg = pd.read_csv('kg.csv')
print(kg.shape)
print(kg.columns.tolist())
print(kg['relation'].value_counts())

/tmp/ipython-input-73293618.py:1: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  kg = pd.read_csv('kg.csv')


(8100498, 12)
['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']
relation
anatomy_protein_present       3036406
drug_drug                     2672628
protein_protein                642150
disease_phenotype_positive     300634
bioprocess_protein             289610
cellcomp_protein               166804
disease_protein                160822
molfunc_protein                139060
drug_effect                    129568
bioprocess_bioprocess          105772
pathway_protein                 85292
disease_disease                 64388
contraindication                61350
drug_protein                    51306
anatomy_protein_absent          39774
phenotype_phenotype             37472
anatomy_anatomy                 28064
molfunc_molfunc                 27148
indication                      18776
cellcomp_cellcomp                9690
phenotype_protein                6660
off-label use                    5136
pathwa

In [ ]:
print(kg['relation'].value_counts())

relation
anatomy_protein_present       3036406
drug_drug                     2672628
protein_protein                642150
disease_phenotype_positive     300634
bioprocess_protein             289610
cellcomp_protein               166804
disease_protein                160822
molfunc_protein                139060
drug_effect                    129568
bioprocess_bioprocess          105772
pathway_protein                 85292
disease_disease                 64388
contraindication                61350
drug_protein                    51306
anatomy_protein_absent          39774
phenotype_phenotype             37472
anatomy_anatomy                 28064
molfunc_molfunc                 27148
indication                      18776
cellcomp_cellcomp                9690
phenotype_protein                6660
off-label use                    5136
pathway_pathway                  5070
exposure_disease                 4608
exposure_exposure                4140
exposure_bioprocess              3250
exp

In [ ]:
# kg.csv里x_type和y_type的组合
type_combos = kg.groupby(['x_type', 'y_type']).size().reset_index(name='count')
print(type_combos.sort_values('count', ascending=False))

                x_type              y_type    count
14                drug                drug  2672628
1              anatomy        gene/protein  1538090
27        gene/protein             anatomy  1538090
34        gene/protein        gene/protein   642150
10             disease    effect/phenotype   151510
17    effect/phenotype             disease   151510
28        gene/protein  biological_process   144805
4   biological_process        gene/protein   144805
2   biological_process  biological_process   105772
7   cellular_component        gene/protein    83402
29        gene/protein  cellular_component    83402
12             disease        gene/protein    80411
30        gene/protein             disease    80411
35        gene/protein  molecular_function    69530
38  molecular_function        gene/protein    69530
15                drug    effect/phenotype    64784
18    effect/phenotype                drug    64784
8              disease             disease    64388
40          

In [ ]:
print(kg[kg['relation'] == 'disease_phenotype_positive'].shape[0])
print(kg[kg['relation'] == 'indication'].shape[0])

300634
18776


In [ ]:
# 看一下disease_phenotype_positive的具体行
pheno_edges = kg[kg['relation'] == 'disease_phenotype_positive']
print(pheno_edges[['x_index', 'x_type', 'y_index', 'y_type']].head(10))

# 然后看有没有反向边
reverse = kg[kg['relation'] == 'phenotype_disease_positive']
print(f"\nReverse relation exists: {reverse.shape[0]}")

         x_index   x_type  y_index            y_type
3085246    27472  disease    24442  effect/phenotype
3085247    27158  disease    22784  effect/phenotype
3085248    27158  disease    84344  effect/phenotype
3085249    27158  disease    22488  effect/phenotype
3085250    27158  disease    22272  effect/phenotype
3085251    27158  disease    23462  effect/phenotype
3085252    27158  disease    26674  effect/phenotype
3085253    27158  disease    22770  effect/phenotype
3085254    27158  disease    84589  effect/phenotype
3085255    27158  disease    84343  effect/phenotype

Reverse relation exists: 0


In [ ]:
print(f"Total nodes: {nodes.shape[0]}")
print(f"Total edges: {kg.shape[0]}")
print(f"Total diseases: {nodes[nodes['node_type']=='disease'].shape[0]}")
print(f"Total drugs: {nodes[nodes['node_type']=='drug'].shape[0]}")
print(f"Total phenotypes: {nodes[nodes['node_type']=='effect/phenotype'].shape[0]}")

NameError: name 'nodes' is not defined

In [ ]:
# TxGNN
!git clone https://github.com/mims-harvard/TxGNN.git

fatal: destination path 'TxGNN' already exists and is not an empty directory.


In [ ]:
# PrimeKG自带的disease-phenotype边
pheno_edges = edges[edges['relation'] == 'disease_phenotype_positive']

# indication边
indication_edges = edges[edges['relation'] == 'indication']

# 同时有表型和药物的疾病
disease_with_pheno = set(pheno_edges['x_index'])  # 假设x是disease
disease_with_drug = set(indication_edges['y_index'])  # 假设y是disease

overlap = disease_with_pheno & disease_with_drug
print(len(overlap))
# 539个疾病，每个疾病同时满足两个条件：1. 在PrimeKG里有disease_phenotype_positive边（即有已知的症状）2.在PrimeKG里有indication边（即有已知的有效药物）

NameError: name 'edges' is not defined

In [ ]:
# 从kg.csv
indication_edges = kg[kg['relation'] == 'indication']
pheno_edges = kg[kg['relation'] == 'disease_phenotype_positive']

diseases_with_indications = set(indication_edges['y_index'])
diseases_with_phenotypes = set(pheno_edges['x_index'])
overlap = diseases_with_indications & diseases_with_phenotypes

print(f"Unique diseases with both: {len(overlap)}")

Unique diseases with both: 539


In [ ]:
# 495个diseases里涉及的unique phenotypes
overlap_diseases = diseases_with_indications & diseases_with_phenotypes

pheno_edges_filtered = kg[(kg['relation'] == 'disease_phenotype_positive') &
                          (kg['x_index'].isin(overlap_diseases))]

print(f"Unique phenotypes across all 495 diseases: {pheno_edges_filtered['y_index'].nunique()}")

# 再分train/test看
train_diseases_set = set([d for d, _ in train_pairs])
test_diseases_set = set([d for d, _ in test_pairs])

train_phenos = kg[(kg['relation'] == 'disease_phenotype_positive') &
                  (kg['x_index'].isin(train_diseases_set))]['y_index'].unique()
test_phenos = kg[(kg['relation'] == 'disease_phenotype_positive') &
                 (kg['x_index'].isin(test_diseases_set))]['y_index'].unique()

print(f"Unique phenotypes in train diseases: {len(train_phenos)}")
print(f"Unique phenotypes in test diseases: {len(test_phenos)}")
print(f"Phenotypes in test but not in train: {len(set(test_phenos) - set(train_phenos))}")

Unique phenotypes across all 495 diseases: 3518


NameError: name 'train_pairs' is not defined

In [ ]:
# 获取所有495个diseases的名字
disease_names = nodes[nodes['node_index'].isin(overlap)][['node_index', 'node_name']]
print(disease_names.to_string())

       node_index                                                                   node_name
27219       27219                       hypogonadotropic hypogonadism with or without anosmia
27248       27248                                               amyotrophic lateral sclerosis
27249       27249                                          isolated growth hormone deficiency
27275       27275                                                           neurofibromatosis
27285       27285                                         hereditary hypophosphatemic rickets
27286       27286                                                                galactosemia
27292       27292                                                    glycogen storage disease
27304       27304                                                       mucopolysaccharidosis
27355       27355                                                             Scheie syndrome
27361       27361                                           

In [ ]:
# 统计训练数据规模
indication_subset = indication_edges[indication_edges['y_index'].isin(overlap)]
print(f"可用疾病数: {len(overlap)}")
print(f"疾病-药物训练对数: {len(indication_subset)}")
print(f"平均每个疾病有多少药物: {len(indication_subset)/len(overlap):.1f}")

pheno_subset = pheno_edges[pheno_edges['x_index'].isin(overlap)]
phenotypes_per_disease = pheno_subset.groupby('x_index')['y_index'].nunique()
print(f"平均每个疾病有多少表型: {phenotypes_per_disease.mean():.1f}")
print(f"中位数: {phenotypes_per_disease.median():.1f}")

可用疾病数: 539
疾病-药物训练对数: 3487
平均每个疾病有多少药物: 6.5
平均每个疾病有多少表型: 22.7
中位数: 13.0


In [ ]:
# 检查overlap里的节点是否真的都是disease类型
nodes = pd.read_csv('nodes.csv')  # 或相应文件
overlap_nodes = nodes[nodes['node_index'].isin(overlap)]
print(overlap_nodes['node_type'].value_counts())
# 应该全部是 'disease'

FileNotFoundError: [Errno 2] No such file or directory: 'nodes.csv'

In [ ]:
import torch
import numpy as np

# 建立node index映射
all_node_ids = sorted(nodes['node_index'].unique())
node2idx = {nid: i for i, nid in enumerate(all_node_ids)}

# 把overlap里的疾病对应的phenotype列表建好
disease_phenotypes = {}
for disease_id in overlap:
    phenos = pheno_edges[pheno_edges['x_index'] == disease_id]['y_index'].tolist()
    disease_phenotypes[disease_id] = phenos

# 疾病-药物正样本对
pos_pairs = list(zip(indication_subset['y_index'], indication_subset['x_index']))
# (disease_id, drug_id)

print(f"疾病数: {len(disease_phenotypes)}")
print(f"正样本对数: {len(pos_pairs)}")
print(f"示例: disease={pos_pairs[0][0]}, drug={pos_pairs[0][1]}, phenotypes={disease_phenotypes[pos_pairs[0][0]][:3]}")

疾病数: 539
正样本对数: 3487
示例: disease=33577, drug=16687, phenotypes=[94180, 23513, 94390]


In [ ]:
from sklearn.model_selection import train_test_split

diseases = list(overlap)
train_diseases, test_diseases = train_test_split(diseases, test_size=0.2, random_state=42)

print(f"训练疾病数: {len(train_diseases)}")
print(f"测试疾病数: {len(test_diseases)}")

# 对应的disease-drug pairs
train_pairs = indication_subset[indication_subset['y_index'].isin(train_diseases)]
test_pairs = indication_subset[indication_subset['y_index'].isin(test_diseases)]

print(f"训练pairs数: {len(train_pairs)}")
print(f"测试pairs数: {len(test_pairs)}")

训练疾病数: 431
测试疾病数: 108
训练pairs数: 2969
测试pairs数: 518


In [ ]:
# 495个diseases里涉及的unique phenotypes
overlap_diseases = diseases_with_indications & diseases_with_phenotypes

pheno_edges_filtered = kg[(kg['relation'] == 'disease_phenotype_positive') &
                          (kg['x_index'].isin(overlap_diseases))]

print(f"Unique phenotypes across all 495 diseases: {pheno_edges_filtered['y_index'].nunique()}")

# 再分train/test看
train_diseases_set = set(train_pairs['y_index'].tolist())
test_diseases_set = set(test_pairs['y_index'].tolist())

train_phenos = kg[(kg['relation'] == 'disease_phenotype_positive') &
                  (kg['x_index'].isin(train_diseases_set))]['y_index'].unique()
test_phenos = kg[(kg['relation'] == 'disease_phenotype_positive') &
                 (kg['x_index'].isin(test_diseases_set))]['y_index'].unique()

print(f"Unique phenotypes in train diseases: {len(train_phenos)}")
print(f"Unique phenotypes in test diseases: {len(test_phenos)}")
print(f"Phenotypes in test but not in train: {len(set(test_phenos) - set(train_phenos))}")

Unique phenotypes across all 495 diseases: 3518
Unique phenotypes in train diseases: 3260
Unique phenotypes in test diseases: 1177
Phenotypes in test but not in train: 258


In [ ]:
# test diseases的phenotype统计
test_pheno = pheno_subset[pheno_subset['x_index'].isin(test_diseases)]
test_phenotypes_per_disease = test_pheno.groupby('x_index')['y_index'].nunique()
print(f"test平均phenotypes: {test_phenotypes_per_disease.mean():.1f}")
print(f"test中位数: {test_phenotypes_per_disease.median():.1f}")

test平均phenotypes: 19.1
test中位数: 13.0


In [ ]:
# 验证两个文件的relation counts是否一样
print(edges['relation'].value_counts()['disease_phenotype_positive'])
print(kg['relation'].value_counts()['disease_phenotype_positive'])

300634
300634


In [ ]:
pheno_edges_kg = kg[kg['relation'] == 'disease_phenotype_positive']
indication_edges_kg = kg[kg['relation'] == 'indication']

disease_with_pheno_kg = set(pheno_edges_kg['x_index'])
disease_with_drug_kg = set(indication_edges_kg['y_index'])

overlap_kg = disease_with_pheno_kg & disease_with_drug_kg
print(len(overlap_kg))

539


In [ ]:
indication_kg = kg[kg['relation'] == 'indication']
print(indication_kg[['x_type', 'y_type']].value_counts())

x_type  y_type 
drug    disease    9388
Name: count, dtype: int64


In [ ]:
# 用kg.csv重新算
pheno_edges_kg = kg[kg['relation'] == 'disease_phenotype_positive']
indication_edges_kg = kg[kg['relation'] == 'indication']

disease_with_pheno_kg = set(pheno_edges_kg['x_index'])
disease_with_drug_kg = set(indication_edges_kg['y_index'])

overlap_kg = disease_with_pheno_kg & disease_with_drug_kg

indication_subset_kg = indication_edges_kg[indication_edges_kg['y_index'].isin(overlap_kg)]
pheno_subset_kg = pheno_edges_kg[pheno_edges_kg['x_index'].isin(overlap_kg)]
phenotypes_per_disease_kg = pheno_subset_kg.groupby('x_index')['y_index'].nunique()

print(f"可用疾病数: {len(overlap_kg)}")
print(f"疾病-药物对数: {len(indication_subset_kg)}")
print(f"平均phenotypes: {phenotypes_per_disease_kg.mean():.1f}")
print(f"中位数: {phenotypes_per_disease_kg.median():.1f}")

可用疾病数: 539
疾病-药物对数: 3487
平均phenotypes: 22.7
中位数: 13.0


In [ ]:
# train/test split
import random
random.seed(42)
disease_list_kg = list(overlap_kg)
random.shuffle(disease_list_kg)
split = int(0.8 * len(disease_list_kg))
train_diseases_kg = set(disease_list_kg[:split])
test_diseases_kg = set(disease_list_kg[split:])

train_pairs_kg = indication_subset_kg[indication_subset_kg['y_index'].isin(train_diseases_kg)]
test_pairs_kg = indication_subset_kg[indication_subset_kg['y_index'].isin(test_diseases_kg)]

print(f"训练疾病: {len(train_diseases_kg)}, 测试疾病: {len(test_diseases_kg)}")
print(f"训练pairs: {len(train_pairs_kg)}, 测试pairs: {len(test_pairs_kg)}")

# test diseases phenotype统计
test_pheno_kg = pheno_subset_kg[pheno_subset_kg['x_index'].isin(test_diseases_kg)]
test_phenotypes_per_disease = test_pheno_kg.groupby('x_index')['y_index'].nunique()
print(f"test平均phenotypes: {test_phenotypes_per_disease.mean():.1f}")
print(f"test中位数: {test_phenotypes_per_disease.median():.1f}")

训练疾病: 431, 测试疾病: 108
训练pairs: 2797, 测试pairs: 690
test平均phenotypes: 24.9
test中位数: 16.0


In [ ]:
# test diseases的true label drugs
test_true_drugs = set(test_pairs_kg['x_index'])

# train里出现过的drugs
train_true_drugs = set(train_pairs_kg['x_index'])

# overlap
overlap_drugs = test_true_drugs & train_true_drugs
print(f"test true drugs总数: {len(test_true_drugs)}")
print(f"其中在train里出现过的: {len(overlap_drugs)}")
print(f"比例: {len(overlap_drugs)/len(test_true_drugs):.1%}")

test true drugs总数: 422
其中在train里出现过的: 289
比例: 68.5%


In [ ]:
print(kg.columns.tolist())

['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source', 'y_index', 'y_id', 'y_type', 'y_name', 'y_source']


In [ ]:
import torch
import torch.nn as nn
# Proof of concept: 用phenotype embedding均值预测药物
# 不需要GNN，直接用随机初始化embedding测逻辑是否跑通

hidden_dim = 64
num_nodes = len(all_node_ids)

# 随机初始化embedding（后面换成训练过的）
torch.manual_seed(42)
embeddings = nn.Embedding(num_nodes, hidden_dim)

# 所有药物的index
drug_nodes = nodes[nodes['node_type'] == 'drug']['node_index'].tolist()
drug_tensor = torch.tensor(drug_nodes)

# 评估函数
def evaluate(embeddings, disease_phenotypes, pos_pairs, k=10):
    hits = 0
    with torch.no_grad():
        drug_embs = embeddings(drug_tensor)  # [num_drugs, d]

        for disease_id, drug_id in pos_pairs:
            phenos = disease_phenotypes[disease_id]
            pheno_tensor = torch.tensor(phenos)

            # phenotype embedding取均值
            pheno_emb = embeddings(pheno_tensor).mean(0)  # [d]

            # 跟所有药物算相似度
            scores = (drug_embs * pheno_emb).sum(-1)  # [num_drugs]

            # 正确药物排第几
            drug_rank = (scores >= scores[drug_nodes.index(drug_id)]).sum().item()
            if drug_rank <= k:
                hits += 1

    return hits / len(pos_pairs)

recall = evaluate(embeddings, disease_phenotypes, pos_pairs, k=10)
random_baseline = 10 / len(drug_nodes)
print(f"Random baseline Recall@10: {random_baseline:.4f}")
print(f"当前模型 Recall@10: {recall:.4f}")

Random baseline Recall@10: 0.0013
当前模型 Recall@10: 0.0009


In [ ]:
def evaluate(embeddings, disease_phenotypes, pos_pairs, k=10):
    hits = 0
    device = next(embeddings.parameters()).device
    with torch.no_grad():
        drug_embs = embeddings(drug_nodes_t)  # 用已经在GPU上的drug_nodes_t

        for disease_id, drug_id in pos_pairs:
            phenos = torch.tensor(disease_phenotypes[disease_id], dtype=torch.long).to(device)
            pheno_emb = embeddings(phenos).mean(0)
            scores = (drug_embs * pheno_emb).sum(-1)
            drug_pos = (drug_nodes_t == drug_id).nonzero(as_tuple=True)[0]
            drug_rank = (scores >= scores[drug_pos]).sum().item()
            if drug_rank <= k:
                hits += 1
    return hits / len(pos_pairs)

device = torch.device('cuda')
embeddings = nn.Embedding(num_nodes, 64).to(device)
optimizer = torch.optim.Adam(embeddings.parameters(), lr=0.01)

# 预处理成固定长度（padding）
max_pheno = max(len(v) for v in disease_phenotypes.values())
batch_size = 256

# 把所有训练对转成tensor
pheno_padded = []
pheno_masks = []
for disease_id, _ in pos_pairs:
    phenos = disease_phenotypes[disease_id]
    pad = phenos + [0] * (max_pheno - len(phenos))
    mask = [1]*len(phenos) + [0]*(max_pheno - len(phenos))
    pheno_padded.append(pad)
    pheno_masks.append(mask)

pheno_tensor_all = torch.tensor(pheno_padded, dtype=torch.long).to(device)
mask_tensor_all = torch.tensor(pheno_masks, dtype=torch.float).to(device)
drug_tensor_all = torch.tensor([d for _, d in pos_pairs], dtype=torch.long).to(device)
drug_nodes_t = torch.tensor(drug_nodes, dtype=torch.long).to(device)

for epoch in range(20):
    total_loss = 0
    perm = torch.randperm(len(pos_pairs))

    for i in range(0, len(pos_pairs), batch_size):
        idx = perm[i:i+batch_size]

        phenos = pheno_tensor_all[idx]        # [B, max_pheno]
        masks = mask_tensor_all[idx]          # [B, max_pheno]
        drugs = drug_tensor_all[idx]          # [B]

        # masked mean pooling
        pheno_emb = embeddings(phenos)        # [B, max_pheno, 64]
        pheno_emb = (pheno_emb * masks.unsqueeze(-1)).sum(1) / masks.sum(1, keepdim=True)

        # 负采样
        neg_idx = torch.randint(len(drug_nodes), (len(idx),), device=device)
        neg_drugs = drug_nodes_t[neg_idx]

        pos_emb = embeddings(drugs)           # [B, 64]
        neg_emb = embeddings(neg_drugs)       # [B, 64]

        pos_score = (pheno_emb * pos_emb).sum(-1)
        neg_score = (pheno_emb * neg_emb).sum(-1)

        loss = F.relu(1.0 - pos_score + neg_score).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/20, Loss: {total_loss:.4f}")
    if (epoch+1) % 5 == 0:
        recall = evaluate(embeddings, disease_phenotypes, pos_pairs, k=10)
        print(f"  → Recall@10: {recall:.4f} (random: {random_baseline:.4f})")

Epoch 1/20, Loss: 34.3681
Epoch 2/20, Loss: 26.4562
Epoch 3/20, Loss: 21.6672
Epoch 4/20, Loss: 16.9316
Epoch 5/20, Loss: 13.9206
  → Recall@10: 0.0072 (random: 0.0013)
Epoch 6/20, Loss: 11.3365
Epoch 7/20, Loss: 8.7966
Epoch 8/20, Loss: 6.9925
Epoch 9/20, Loss: 6.2608
Epoch 10/20, Loss: 5.4346
  → Recall@10: 0.0261 (random: 0.0013)
Epoch 11/20, Loss: 4.2037
Epoch 12/20, Loss: 3.4856
Epoch 13/20, Loss: 3.1342
Epoch 14/20, Loss: 2.6198
Epoch 15/20, Loss: 2.3199
  → Recall@10: 0.0602 (random: 0.0013)
Epoch 16/20, Loss: 1.8402
Epoch 17/20, Loss: 1.7064
Epoch 18/20, Loss: 1.7286
Epoch 19/20, Loss: 1.4081
Epoch 20/20, Loss: 1.2965
  → Recall@10: 0.1035 (random: 0.0013)


In [ ]:
# 重新初始化embedding
embeddings = nn.Embedding(num_nodes, 64).to(device)
optimizer = torch.optim.Adam(embeddings.parameters(), lr=0.01)

# 重新预处理，只用train_pairs
pheno_padded = []
pheno_masks = []
for disease_id, _ in train_pairs:
    phenos = disease_phenotypes[disease_id]
    pad = phenos + [0] * (max_pheno - len(phenos))
    mask = [1]*len(phenos) + [0]*(max_pheno - len(phenos))
    pheno_padded.append(pad)
    pheno_masks.append(mask)

pheno_tensor_train = torch.tensor(pheno_padded, dtype=torch.long).to(device)
mask_tensor_train = torch.tensor(pheno_masks, dtype=torch.float).to(device)
drug_tensor_train = torch.tensor([d for _, d in train_pairs], dtype=torch.long).to(device)

for epoch in range(20):
    total_loss = 0
    perm = torch.randperm(len(train_pairs))

    for i in range(0, len(train_pairs), batch_size):
        idx = perm[i:i+batch_size]
        phenos = pheno_tensor_train[idx]
        masks = mask_tensor_train[idx]
        drugs = drug_tensor_train[idx]

        pheno_emb = embeddings(phenos)
        pheno_emb = (pheno_emb * masks.unsqueeze(-1)).sum(1) / masks.sum(1, keepdim=True)

        neg_idx = torch.randint(len(drug_nodes), (len(idx),), device=device)
        neg_drugs = drug_nodes_t[neg_idx]

        pos_emb = embeddings(drugs)
        neg_emb = embeddings(neg_drugs)

        loss = F.relu(1.0 - (pheno_emb * pos_emb).sum(-1) + (pheno_emb * neg_emb).sum(-1)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/20, Loss: {total_loss:.4f}")

# 最终在test_pairs上评估
test_recall = evaluate(embeddings, disease_phenotypes, test_pairs, k=10)
train_recall = evaluate(embeddings, disease_phenotypes, train_pairs, k=10)
print(f"\nTrain Recall@10: {train_recall:.4f}")
print(f"Test Recall@10:  {test_recall:.4f}")
print(f"Random baseline: {random_baseline:.4f}")

Epoch 1/20, Loss: 30.5012
Epoch 2/20, Loss: 24.9936
Epoch 3/20, Loss: 19.2492
Epoch 4/20, Loss: 16.0310
Epoch 5/20, Loss: 13.1620
Epoch 6/20, Loss: 10.2599
Epoch 7/20, Loss: 8.6409
Epoch 8/20, Loss: 7.3017
Epoch 9/20, Loss: 6.0219
Epoch 10/20, Loss: 5.6842
Epoch 11/20, Loss: 4.5291
Epoch 12/20, Loss: 3.0679
Epoch 13/20, Loss: 3.3824
Epoch 14/20, Loss: 2.6697
Epoch 15/20, Loss: 2.8394
Epoch 16/20, Loss: 2.2248
Epoch 17/20, Loss: 1.9269
Epoch 18/20, Loss: 1.6766
Epoch 19/20, Loss: 1.5049
Epoch 20/20, Loss: 1.4552

Train Recall@10: 0.0706
Test Recall@10:  0.0172
Random baseline: 0.0013


In [ ]:
# test diseases的true label drugs
test_true_drugs = set(test_pairs_kg['x_index'])

# train里出现过的drugs
train_true_drugs = set(train_pairs_kg['x_index'])

# overlap
overlap_drugs = test_true_drugs & train_true_drugs
print(f"test true drugs总数: {len(test_true_drugs)}")
print(f"其中在train里出现过的: {len(overlap_drugs)}")
print(f"比例: {len(overlap_drugs)/len(test_true_drugs):.1%}")

NameError: name 'test_pairs_kg' is not defined